In [1]:
# Parameters
run_id = "28b2de94-0f48-40bf-92ca-85995e1ba8da"
artifacts_dir = "/home/adnoman/projects/aml_gan/AMLend2end/artifacts/runs/28b2de94-0f48-40bf-92ca-85995e1ba8da"
sample_size = None
epochs = None
threshold = None


### Create training dataset for anomaly detection model
In this notebook We are going to create training dataset from node embeddings feature group and register to Hopsworks Feature Store. 
![Training Dataset](./images/create_training_dataset.png)

### Create a connection to hsfs

In [2]:
# Setup for local execution
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
OUTPUT_PATH = os.path.join(BASE_PATH, "output")
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")

print(f"Output path: {OUTPUT_PATH}")
print(f"Training data path: {TRAINING_DATA_PATH}")

Output path: /home/adnoman/projects/aml_gan/AMLend2end/output
Training data path: /home/adnoman/projects/aml_gan/AMLend2end/training_data


### Retrieve alert nodes feature group from hsfs

In [3]:
# Load node embeddings feature group (created in notebook 5)
node_embeddings_fg = pd.read_parquet(os.path.join(OUTPUT_PATH, "node_embeddings_fg.parquet"))

print(f"Loaded node embeddings: {node_embeddings_fg.shape}")
print(f"Columns: {node_embeddings_fg.columns.tolist()[:5]}... + is_sar")
node_embeddings_fg.head()

Loaded node embeddings: (7347, 34)
Columns: ['id', 'emb_0', 'emb_1', 'emb_2', 'emb_3']... + is_sar


,id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31,is_sar
0,3aa9646b,-0.008853,0.004043,-0.003818,0.007575,0.023354,-0.027285,-0.005630,0.025699,0.025976,...,0.029562,-0.011895,0.020304,-0.001925,0.007217,-0.029996,0.016181,0.015940,0.008504,0
1,1e46e726,0.022349,-0.001167,0.030925,-0.012870,0.030629,0.014389,0.023963,0.009350,-0.012923,...,-0.005137,0.017466,0.022701,-0.026283,0.017401,0.015713,-0.000234,-0.007488,0.008504,0
2,49203bc3,-0.005501,-0.009205,-0.027755,-0.014818,-0.012859,-0.024729,0.023249,0.016355,0.025465,...,-0.015878,-0.013799,-0.000237,0.019603,0.026605,-0.015244,0.002540,0.007230,0.006761,0
3,a74d1101,0.011437,-0.000023,0.007053,-0.003801,0.022929,0.025165,-0.000650,-0.014209,-0.006374,...,-0.027126,0.013434,0.008401,-0.026911,-0.028524,0.011721,0.026398,-0.003113,-0.029955,1
4,616d4505,-0.017530,-0.002801,-0.000988,-0.024782,-0.026430,-0.030293,0.005277,0.004350,0.028420,...,0.026174,-0.028986,-0.003035,-0.015092,0.024190,0.002712,0.019147,0.031202,-0.002625,0


### Prepare training datasets for anomaly detection 
###### In the next notebook we are going to train [gan for anomaly detection](https://arxiv.org/pdf/1905.11034.pdf). Durring training step  we will provide only features of accounts that have never been reported for money laundering behaviour.  But we will disclose previously reported accounts to the model only in evaluation step.   

In [4]:
# Get embedding columns
emb_cols = [c for c in node_embeddings_fg.columns if c.startswith('emb_')]
print(f"Embedding dimensions: {len(emb_cols)}")

# Filter non-SAR nodes for training (is_sar == 0)
non_sar_df = node_embeddings_fg[node_embeddings_fg['is_sar'] == 0].copy()
print(f"Non-SAR nodes: {len(non_sar_df)}")

Embedding dimensions: 32
Non-SAR nodes: 6531


In [5]:
# Preview non-SAR embeddings
non_sar_df[emb_cols[:5] + ['is_sar']].head()

,emb_0,emb_1,emb_2,emb_3,emb_4,is_sar
0,-0.008853,0.004043,-0.003818,0.007575,0.023354,0
1,0.022349,-0.001167,0.030925,-0.012870,0.030629,0
2,-0.005501,-0.009205,-0.027755,-0.014818,-0.012859,0
4,-0.017530,-0.002801,-0.000988,-0.024782,-0.026430,0
6,0.009941,0.001355,-0.030964,-0.017995,-0.014444,0


In [6]:
print(f"Non-SAR count: {len(non_sar_df)}")

Non-SAR count: 6531


In [7]:
# Create train/test split for non-SAR data (80/20 split)
non_sar_train, non_sar_test = train_test_split(
    non_sar_df, 
    test_size=0.2, 
    random_state=42
)

print(f"Non-SAR train: {len(non_sar_train)}")
print(f"Non-SAR test: {len(non_sar_test)}")

# Extract embeddings as numpy arrays
X_train = non_sar_train[emb_cols].values
y_train = non_sar_train['is_sar'].values

X_test_non_sar = non_sar_test[emb_cols].values
y_test_non_sar = non_sar_test['is_sar'].values

print(f"\nTraining data shape: {X_train.shape}")
print(f"Test data shape: {X_test_non_sar.shape}")

Non-SAR train: 5224
Non-SAR test: 1307

Training data shape: (5224, 32)
Test data shape: (1307, 32)


## For testing and evaluation we will include known SAR nodes to measure anomaly score  

In [8]:
# For evaluation, we need both SAR and non-SAR test data
# Get SAR nodes
sar_df = node_embeddings_fg[node_embeddings_fg['is_sar'] == 1].copy()
print(f"SAR nodes: {len(sar_df)}")

SAR nodes: 816


In [9]:
# Extract SAR embeddings
X_sar = sar_df[emb_cols].values
y_sar = sar_df['is_sar'].values

print(f"SAR data shape: {X_sar.shape}")

SAR data shape: (816, 32)


In [10]:
# Create evaluation dataset: combine non-SAR test + all SAR nodes
X_eval = np.vstack([X_test_non_sar, X_sar])
y_eval = np.concatenate([y_test_non_sar, y_sar])

print(f"Evaluation dataset shape: {X_eval.shape}")
print(f"Evaluation labels: {len(y_eval)} (SAR: {y_eval.sum()}, Non-SAR: {(y_eval==0).sum()})")

Evaluation dataset shape: (2123, 32)
Evaluation labels: 2123 (SAR: 816, Non-SAR: 1307)


In [11]:
print(f"Non-SAR test count: {len(X_test_non_sar)}")

Non-SAR test count: 1307


In [12]:
print(f"SAR count: {len(X_sar)}")

SAR count: 816


In [13]:
print(f"Evaluation total count: {len(X_eval)}")

Evaluation total count: 2123


In [14]:
# Save training datasets locally (replaces hsfs tfrecord)
GAN_DATA_PATH = os.path.join(TRAINING_DATA_PATH, "gan")
os.makedirs(GAN_DATA_PATH, exist_ok=True)

# Save as numpy arrays (efficient for training)
np.save(os.path.join(GAN_DATA_PATH, "X_train.npy"), X_train)
np.save(os.path.join(GAN_DATA_PATH, "y_train.npy"), y_train)
np.save(os.path.join(GAN_DATA_PATH, "X_eval.npy"), X_eval)
np.save(os.path.join(GAN_DATA_PATH, "y_eval.npy"), y_eval)

print(f"Saved training data to: {GAN_DATA_PATH}")
print(f"  - X_train.npy: {X_train.shape}")
print(f"  - y_train.npy: {y_train.shape}")
print(f"  - X_eval.npy: {X_eval.shape}")
print(f"  - y_eval.npy: {y_eval.shape}")

Saved training data to: /home/adnoman/projects/aml_gan/AMLend2end/training_data/gan
  - X_train.npy: (5224, 32)
  - y_train.npy: (5224,)
  - X_eval.npy: (2123, 32)
  - y_eval.npy: (2123,)


## Training dataset provenance
![Training dataset provenance](./images/provenance_td.png)

In [15]:
# Summary
print("=" * 50)
print("GAN Training Datasets Created")
print("=" * 50)
print(f"Training set (non-SAR only): {X_train.shape[0]} samples")
print(f"Evaluation set (SAR + non-SAR): {X_eval.shape[0]} samples")
print(f"  - Non-SAR: {(y_eval==0).sum()}")
print(f"  - SAR: {y_eval.sum()}")
print(f"Embedding dimensions: {X_train.shape[1]}")
print(f"\nSaved to: {GAN_DATA_PATH}")
print("=" * 50)

GAN Training Datasets Created
Training set (non-SAR only): 5224 samples
Evaluation set (SAR + non-SAR): 2123 samples
  - Non-SAR: 1307
  - SAR: 816
Embedding dimensions: 32

Saved to: /home/adnoman/projects/aml_gan/AMLend2end/training_data/gan
